# C389 CA1 — T4: Wait or cross, with Gradio deployment

**Run sections 1–11 in order to start the demo.** One team app can process different videos:
upload a video, click four waiting-area corners in order around its edge, click two crossing-line
endpoints, then process. Mark ground under people's feet, not their whole bodies. Place the line
across the route where you want a crossing to be counted.

The demo shows the annotated video, wait/cross counts, and **Track ID | Waiting time (s)**.
Waiting means observed **stationary time inside the selected area**, using your current
**1-second** rule. Several qualifying waits for the same ID are added together. A shorter pause
does not contribute. `>=` indicates a partial observed wait; it is not an exact full duration.
Tracking IDs are kept unchanged and can have gaps or split when tracking is lost.

Sections 12–16 and 19 are optional review/evaluation tools after a demo run. The old parameter
sweep and freeze sections 17–18 are removed. Each run automatically saves its settings and
evidence instead. This notebook uses your existing trained model; it does not retrain it.


## 1. Install libraries
In Colab, select a GPU runtime first. This installs the model library version used in your
current notebook and a pinned Gradio version. If asked to restart after installation, restart
the runtime and continue from section 2.


In [ ]:
%pip -q install ultralytics==8.4.120 gradio==6.31.0 imageio-ffmpeg==0.6.0


## 2. Rules and saving helpers
These definitions are used by the app; no video output is expected yet.


In [ ]:
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path
import csv
import hashlib
import json
import math
import statistics
import time


@dataclass(frozen=True)
class Rules:
    min_wait_seconds: float = 3.0
    motion_window_seconds: float = 0.75
    smoothing_seconds: float = 0.20
    still_body_heights_per_second: float = 0.06
    max_observation_gap_seconds: float = 0.25
    min_track_seconds: float = 0.5
    min_track_observations: int = 5
    line_margin_body_heights: float = 0.03
    line_confirm_seconds: float = 0.15

    def validate(self):
        for key, value in asdict(self).items():
            if not math.isfinite(value) or value <= 0:
                raise ValueError(f'{key} must be finite and greater than zero.')
        if self.min_wait_seconds < self.motion_window_seconds:
            raise ValueError('Minimum wait must be at least the motion window.')
        if self.min_track_observations != int(self.min_track_observations):
            raise ValueError('min_track_observations must be a whole number.')


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')


def save_csv(path, rows, fields=None):
    rows = list(rows)
    fields = fields or (list(rows[0]) if rows else [])
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: json.dumps(v) if isinstance(v, (list, dict)) else v
                             for k, v in row.items()})


## 3. Ground-area and crossing-line checks
The app uses the points selected on the uploaded video's prepared frame.


In [ ]:
def point_in_polygon(point, polygon):
    """Boundary is included. Polygon is in frame pixels, ordered round its edge."""
    x, y = point
    inside = False
    for a, b in zip(polygon, polygon[1:] + polygon[:1]):
        ax, ay = a
        bx, by = b
        cross = (bx - ax) * (y - ay) - (by - ay) * (x - ax)
        if abs(cross) < 1e-7 and min(ax, bx) <= x <= max(ax, bx) and min(ay, by) <= y <= max(ay, by):
            return True
        if (ay > y) != (by > y):
            hit = ax + (y - ay) * (bx - ax) / (by - ay)
            if x < hit:
                inside = not inside
    return inside


def line_distance(point, line):
    (ax, ay), (bx, by) = line
    length = math.hypot(bx - ax, by - ay)
    if not length:
        raise ValueError('Crossing-line endpoints must differ.')
    return ((bx - ax) * (point[1] - ay) - (by - ay) * (point[0] - ax)) / length


def finite_line_crossing(a, b, line):
    """Return interpolation fraction when the motion crosses the FINITE segment."""
    da, db = line_distance(a, line), line_distance(b, line)
    if da * db > 0 or abs(da - db) < 1e-12:
        return None
    fraction = da / (da - db)
    px, py = a[0] + fraction * (b[0] - a[0]), a[1] + fraction * (b[1] - a[1])
    (ax, ay), (bx, by) = line
    projection = ((px - ax) * (bx - ax) + (py - ay) * (by - ay)) / ((bx - ax)**2 + (by - ay)**2)
    return fraction if 0 <= fraction <= 1 and 0 <= projection <= 1 else None


def validate_site(site, width, height):
    if site.get('frame_width') != width or site.get('frame_height') != height:
        raise ValueError('Site resolution differs from the video. Mark this camera setup again.')
    area, line = site.get('waiting_area'), site.get('crossing_line')
    if not area or len(area) < 3 or not line or len(line) != 2:
        raise ValueError('Set waiting_area corners and two crossing_line endpoints from your frame.')
    for point in list(area) + list(line):
        if len(point) != 2 or not all(math.isfinite(v) for v in point):
            raise ValueError('Every site point must contain two finite coordinates.')
        if not (0 <= point[0] < width and 0 <= point[1] < height):
            raise ValueError(f'Point {point} lies outside this frame.')
    signed_area = sum(a[0]*b[1] - b[0]*a[1] for a,b in zip(area, area[1:] + area[:1]))
    if abs(signed_area) < 1:
        raise ValueError('Waiting area has zero area or invalid corner order.')
    line_distance((0, 0), line)
    if site.get('crossing_direction') not in ('both', 'negative_to_positive', 'positive_to_negative'):
        raise ValueError('Choose a crossing_direction: both, negative_to_positive, positive_to_negative.')
    # Capture notes are useful evidence but are not needed to compute the geometry.


## 4. Per-ID waiting-time calculation
Uses the smoothed bottom centre of each box, movement and timestamps to find stationary waits.


In [ ]:
def _prepare_observations(observations, rules):
    """Causal median smoothing, reset at gaps; no interpolation across missing spans."""
    prepared, history, segment = [], [], 0
    previous = None
    for original in observations:
        row = dict(original)
        t = float(row['seconds'])
        if not math.isfinite(t) or t < 0:
            raise ValueError('Track timestamps must be finite and nonnegative.')
        if previous is not None and t <= previous:
            raise ValueError('Each track must have strictly increasing timestamps.')
        if previous is not None and t - previous > rules.max_observation_gap_seconds + 1e-9:
            history = []
            segment += 1
        box = row['box']
        if len(box) != 4 or not all(math.isfinite(v) for v in box) or box[3] <= box[1] or box[2] <= box[0]:
            raise ValueError('Each observation needs a finite positive-size box.')
        if len(row['foot']) != 2 or not all(math.isfinite(v) for v in row['foot']):
            raise ValueError('Each observation needs a finite foot point.')
        history.append(row)
        history = [r for r in history if t - r['seconds'] <= rules.smoothing_seconds + 1e-9]
        row['smooth_foot'] = [statistics.median(r['foot'][axis] for r in history) for axis in (0,1)]
        row['height'] = box[3] - box[1]
        row['segment'] = segment
        prepared.append(row)
        previous = t
    return prepared


def waiting_episodes(prepared, area, rules):
    """Observe continuous stillness INSIDE the zone; qualify only at minimum duration.

    Motion is the largest displacement from the window's first smoothed foot,
    divided by median box height and elapsed seconds. It is an image-space cue,
    not physical speed. Reversing back to the start cannot erase earlier motion.
    """
    episodes, timeline, window = [], [], []
    active = None
    segment_first = None

    def finish(reason):
        nonlocal active
        if active is not None:
            active['seconds'] = round(active['end_seconds'] - active['start_seconds'], 6)
            active['qualified'] = active['seconds'] + 1e-9 >= rules.min_wait_seconds
            active['end_reason'] = reason
            active['right_censored'] = reason in ('observation_gap', 'track_end')
            active['incomplete'] = active['left_censored'] or active['right_censored']
            episodes.append(active)
            active = None

    for i, row in enumerate(prepared):
        t = row['seconds']
        if i == 0 or row['segment'] != prepared[i-1]['segment']:
            finish('observation_gap')
            window = []
            segment_first = t
        inside = point_in_polygon(row['smooth_foot'], area)
        if not inside:
            finish('zone_exit')
            window = []
            state, motion = 'outside_waiting_area', None
        else:
            window.append(row)
            # Retain a point at or just before the beginning of the window.
            while len(window) > 1 and window[1]['seconds'] <= t - rules.motion_window_seconds:
                window.pop(0)
            elapsed = t - window[0]['seconds']
            if elapsed + 1e-9 < rules.motion_window_seconds:
                state, motion = 'observing', None
            else:
                origin = window[0]['smooth_foot']
                extent = max(math.dist(origin, r['smooth_foot']) for r in window)
                motion = extent / statistics.median(r['height'] for r in window) / elapsed
                if motion <= rules.still_body_heights_per_second:
                    if active is None:
                        active = {'start_seconds': window[0]['seconds'], 'end_seconds': t,
                                  'left_censored': window[0]['seconds'] <= segment_first + 1e-9}
                    active['end_seconds'] = t
                    state = ('waiting' if t - active['start_seconds'] + 1e-9 >= rules.min_wait_seconds
                             else 'possible_wait')
                else:
                    finish('movement')
                    state = 'moving_in_waiting_area'
        timeline.append({'frame': row['frame'], 'seconds': t, 'state': state,
                         'motion_body_heights_per_second': motion,
                         'observed_wait_seconds': round(t - active['start_seconds'], 6) if active else 0.0,
                         'smooth_foot': row['smooth_foot']})
    finish('track_end')
    return episodes, timeline


## 5. Crossing events and per-ID results
Keeps the actual tracker IDs and separates waiting from crossing.


In [ ]:
def crossing_events(prepared, site, rules):
    """Hysteresis and side confirmation reject jitter; repeated events stay visible."""
    events = []
    stable_side, anchor, candidate = None, None, None
    previous_segment = None
    for row in prepared:
        if row['segment'] != previous_segment:
            stable_side, anchor, candidate = None, None, None
            previous_segment = row['segment']
        distance = line_distance(row['smooth_foot'], site['crossing_line'])
        margin = rules.line_margin_body_heights * row['height']
        side = 1 if distance > margin else -1 if distance < -margin else 0
        if side == 0:
            candidate = None
            continue
        if side == stable_side:
            anchor, candidate = row, None
            continue
        if candidate is None or candidate['side'] != side:
            candidate = {'side': side, 'row': row}
        if row['seconds'] - candidate['row']['seconds'] + 1e-9 < rules.line_confirm_seconds:
            continue
        if stable_side is not None:
            fraction = finite_line_crossing(anchor['smooth_foot'], candidate['row']['smooth_foot'],
                                            site['crossing_line'])
            direction = 'negative_to_positive' if stable_side == -1 else 'positive_to_negative'
            if fraction is not None and site['crossing_direction'] in ('both', direction):
                event_t = anchor['seconds'] + fraction * (candidate['row']['seconds'] - anchor['seconds'])
                dx = candidate['row']['smooth_foot'][0] - anchor['smooth_foot'][0]
                events.append({'seconds': round(event_t, 6), 'confirmed_seconds': row['seconds'],
                               'direction': direction,
                               'screen_direction': 'right' if dx > 1e-6 else 'left' if dx < -1e-6 else 'vertical'})
        stable_side, anchor, candidate = side, row, None
    return events


def analyze_tracks(tracks, site, rules):
    rules.validate()
    people, all_episodes, all_events, timelines = [], [], [], {}
    for track_id, observations in sorted(tracks.items(), key=lambda item: int(item[0])):
        if not observations:
            continue
        prepared = _prepare_observations(observations, rules)
        episodes, timeline = waiting_episodes(prepared, site['waiting_area'], rules)
        events = crossing_events(prepared, site, rules)
        track_id = int(track_id)
        for e in episodes:
            e['track_id'] = track_id
        for e in events:
            e['track_id'] = track_id
        waits = [e for e in episodes if e['qualified']]
        eligible = (len(prepared) >= rules.min_track_observations and
                    prepared[-1]['seconds'] - prepared[0]['seconds'] >= rules.min_track_seconds)
        transitions = [(w,e) for w in waits for e in events
                       if w['start_seconds'] + rules.min_wait_seconds <= e['seconds'] + 1e-9]
        if transitions:
            outcome = 'waited_then_crossed'
        elif waits:
            outcome = 'waited_without_later_crossing'
        elif events:
            outcome = 'crossed_without_observed_wait'
        else:
            outcome = 'unresolved'
        flags = []
        if not eligible:
            flags.append('short_track_excluded_from_counts')
        if prepared[-1]['segment']:
            flags.append('observation_gaps')
        if point_in_polygon(prepared[0]['smooth_foot'], site['waiting_area']):
            flags.append('track_begins_inside_waiting_area_prior_behaviour_unknown')
        if any(w['incomplete'] for w in waits):
            flags.append('waiting_duration_is_lower_bound')
        if len(events) > 1:
            flags.append('multiple_crossings_counted_once')
        if waits and events and not transitions:
            flags.append('crossing_precedes_observed_wait')
        majority = Counter(r.get('class_name','unknown') for r in observations).most_common(1)[0][0]
        people.append({'track_id': track_id, 'eligible': eligible, 'outcome': outcome,
                       'waited': bool(waits), 'crossed': bool(events),
                       'observed_wait_seconds': round(sum(w['seconds'] for w in waits), 6),
                       'wait_episode_count': len(waits),
                       'duration_incomplete': any(w['incomplete'] for w in waits),
                       'first_crossing_seconds': events[0]['seconds'] if events else None,
                       'first_seen_seconds': prepared[0]['seconds'], 'last_seen_seconds': prepared[-1]['seconds'],
                       'observations': len(prepared), 'majority_detector_class': majority,
                       'flags': flags})
        timelines[track_id] = timeline
        all_episodes.extend(episodes)
        all_events.extend(events)
    included = [r for r in people if r['eligible']]
    outcomes = Counter(r['outcome'] for r in included)
    summary = {'raw_track_ids': len(people), 'eligible_track_ids': len(included),
               'excluded_short_track_ids': len(people) - len(included),
               'waited_track_ids': sum(r['waited'] for r in included),
               'crossed_track_ids': sum(r['crossed'] for r in included),
               'exclusive_outcomes': {k: outcomes[k] for k in (
                   'waited_then_crossed','waited_without_later_crossing',
                   'crossed_without_observed_wait','unresolved')},
               'count_unit': 'eligible tracker identity; fragmentation can overcount real people',
               'note': 'Waited and crossed overlap. Exclusive outcomes sum to eligible identities.'}
    return {'summary': summary, 'people': people, 'wait_episodes': all_episodes,
            'crossings': all_events, 'timelines': timelines}


## 6. Optional manual evaluation functions
Defines evaluation helpers for section 16; it does not claim performance without labels.


In [ ]:
def parse_bool(value, field):
    if str(value).strip().lower() in ('1','true','yes'):
        return True
    if str(value).strip().lower() in ('0','false','no'):
        return False
    raise ValueError(f'{field}: use true or false; do not leave an unlabelled value.')


def evaluate_people(result, truth_rows, mapping_rows):
    """Manual full-population matching, including missed humans and unmatched tracks.

    Raw system count errors retain fragmentation. Person-grouped classification
    is labelled separately; it never substitutes grouped counts for raw counts.
    Exact duration MAE excludes censored/fragmented cases and reports coverage.
    """
    if not truth_rows:
        raise ValueError('Ground truth is empty: annotate every visible relevant person first.')
    truth = {}
    for raw in truth_rows:
        human = str(raw['human_id']).strip()
        if not human or human in truth:
            raise ValueError('Ground-truth human_id values must be nonempty and unique.')
        waited = parse_bool(raw['waited'], 'waited')
        crossed = parse_bool(raw['crossed'], 'crossed')
        complete = parse_bool(raw['duration_complete'], 'duration_complete')
        duration = None if str(raw.get('wait_seconds','')).strip() == '' else float(raw['wait_seconds'])
        if duration is not None and (not math.isfinite(duration) or duration < 0):
            raise ValueError('wait_seconds must be nonnegative and finite.')
        if waited and complete and duration is None:
            raise ValueError('A complete true wait requires hand-measured wait_seconds.')
        truth[human] = {'waited': waited, 'crossed': crossed,
                        'wait_seconds': duration, 'duration_complete': complete}
    predicted = {int(r['track_id']): r for r in result['people'] if r['eligible']}
    all_ids = {int(r['track_id']) for r in result['people']}
    mapping = {human: [] for human in truth}
    assigned = set()
    for row in mapping_rows:
        human = str(row['human_id']).strip()
        if human not in truth:
            raise ValueError(f'Unknown human_id in mapping: {human}')
        raw_id = str(row.get('track_id','')).strip()
        if not raw_id:
            continue
        tid = int(raw_id)
        if tid not in all_ids or tid in assigned:
            raise ValueError('Mapped track IDs must exist and may belong to only one human.')
        assigned.add(tid)
        mapping[human].append(tid)
    rows, duration_errors, exclusions = [], [], []
    for human, target in truth.items():
        matched = [predicted[t] for t in mapping[human] if t in predicted]
        pw, pc = any(r['waited'] for r in matched), any(r['crossed'] for r in matched)
        exact = (target['waited'] and target['duration_complete'] and len(matched) == 1 and
                 matched[0]['waited'] and not matched[0]['duration_incomplete'])
        error = abs(matched[0]['observed_wait_seconds'] - target['wait_seconds']) if exact else None
        if exact:
            duration_errors.append(error)
        elif target['waited']:
            exclusions.append({'human_id': human, 'reason': 'missed wait, incomplete duration, or fragmented mapping'})
        rows.append({'human_id': human, 'mapped_track_ids': mapping[human],
                     'detected': bool(matched), 'truth_waited': target['waited'], 'predicted_waited': pw,
                     'truth_crossed': target['crossed'], 'predicted_crossed': pc,
                     'classification_correct': bool(matched) and pw == target['waited'] and pc == target['crossed'],
                     'duration_absolute_error_seconds': error})
    unmatched = [r for t,r in predicted.items() if t not in assigned]
    metrics = {'ground_truth_people': len(truth),
               'people_with_eligible_track': sum(r['detected'] for r in rows),
               'unmatched_eligible_track_ids': [r['track_id'] for r in unmatched],
               'humans_with_multiple_track_ids': sum(len(ids) > 1 for ids in mapping.values()),
               'person_grouped_classification_accuracy': sum(r['classification_correct'] for r in rows) / len(rows),
               'duration_mae_seconds': statistics.mean(duration_errors) if duration_errors else None,
               'duration_evaluated_people': len(duration_errors),
               'ground_truth_waiting_people': sum(t['waited'] for t in truth.values()),
               'duration_exclusions': exclusions}
    for name in ('waited','crossed'):
        true_count = sum(t[name] for t in truth.values())
        raw_count = result['summary'][f'{name}_track_ids']
        tp = sum(r[f'truth_{name}'] and r[f'predicted_{name}'] for r in rows)
        fp = sum(not r[f'truth_{name}'] and r[f'predicted_{name}'] for r in rows) + sum(r[name] for r in unmatched)
        fn = sum(r[f'truth_{name}'] and not r[f'predicted_{name}'] for r in rows)
        metrics[f'{name}_truth_count'] = true_count
        metrics[f'{name}_raw_predicted_count'] = raw_count
        metrics[f'{name}_raw_count_absolute_error'] = abs(raw_count - true_count)
        metrics[f'{name}_person_grouped_precision'] = tp / (tp+fp) if tp+fp else None
        metrics[f'{name}_person_grouped_recall'] = tp / (tp+fn) if tp+fn else None
        metrics[f'{name}_person_grouped_f1'] = 2*tp / (2*tp+fp+fn) if 2*tp+fp+fn else None
    return {'metrics': metrics, 'people': rows}


## 7. Video loading and YOLO tracking
Loads a fresh tracker for each video and retains person_left/person_right labels.


In [ ]:
def video_info(path):
    import cv2
    cap = cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened():
            raise ValueError(f'Cannot open video: {path}')
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if not math.isfinite(fps) or fps <= 0 or frames <= 0 or min(width,height) <= 0:
            raise ValueError('Video has invalid FPS, frame count, or resolution. Export constant-frame-rate MP4.')
        return {'fps': fps, 'frames': frames, 'width': width, 'height': height,
                'duration_seconds': frames/fps,
                'timing_basis': 'frame index / container FPS; requires constant frame rate'}
    finally:
        cap.release()


def read_video_frame(path, frame=0):
    import cv2
    cap = cv2.VideoCapture(str(path))
    try:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame)
        ok, image = cap.read()
        if not ok:
            raise ValueError(f'Cannot decode frame {frame}.')
        return image
    finally:
        cap.release()


def track_video(clip_path, model_path, tracker_path, confidence=0.10, image_size=640, progress=None):
    import cv2
    from ultralytics import YOLO
    if not Path(model_path).is_file():
        raise FileNotFoundError('Trained best.pt is missing. No automatic stock/sample fallback is used.')
    info = video_info(clip_path)
    model = YOLO(str(model_path))  # Fresh object: identities cannot leak from another clip.
    names = model.names
    class_ids = [int(i) for i,name in names.items() if name in ('person_left','person_right','person')]
    if not class_ids:
        raise ValueError(f'Model contains no recognised person classes: {names}')
    cap, tracks, decoded, started = cv2.VideoCapture(str(clip_path)), {}, 0, time.perf_counter()
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            result = model.track(frame, persist=True, tracker=str(tracker_path), classes=class_ids,
                                 conf=confidence, imgsz=image_size, verbose=False)[0]
            boxes = result.boxes
            if boxes is not None and boxes.id is not None:
                for box,tid,cid,conf in zip(boxes.xyxy.tolist(), boxes.id.int().tolist(),
                                           boxes.cls.int().tolist(), boxes.conf.tolist()):
                    left,top,right,bottom = box
                    tracks.setdefault(int(tid), []).append({'frame': decoded, 'seconds': decoded/info['fps'],
                        'box': box, 'foot': [(left+right)/2, bottom],
                        'class_name': names[int(cid)], 'confidence': float(conf)})
            decoded += 1
            if progress and decoded % max(1,round(info['fps']*5)) == 0:
                progress(decoded, info['frames'])
    finally:
        cap.release()
    if decoded != info['frames']:
        raise RuntimeError(f'Decoded {decoded}/{info["frames"]} frames. Repair/re-export the video before evaluation.')
    elapsed = time.perf_counter() - started
    return {'clip': str(clip_path), 'model': str(model_path), 'clip_info': info, 'tracks': tracks,
            'class_names': names, 'decoded_frames': decoded, 'processing_seconds': elapsed,
            'processing_fps': decoded / elapsed if elapsed else None}


## 8. Annotated video
Each box shows its ID, detector class, state and observed waiting time.


In [ ]:
def render_video(clip_path, tracked, result, site, rules, output_path):
    """Offline annotated replay. Uses past observations only for displayed states."""
    import cv2
    import numpy as np
    info = tracked['clip_info']
    frames = {}
    for tid, observations in tracked['tracks'].items():
        tid = int(tid)
        states = {r['frame']: r for r in result['timelines'][tid]}
        for obs in observations:
            frames.setdefault(obs['frame'], []).append((tid, obs, states[obs['frame']]))
    eligible = {r['track_id'] for r in result['people'] if r['eligible']}
    cap = cv2.VideoCapture(str(clip_path))
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'),
                             info['fps'], (info['width'],info['height']))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError('Video encoder could not open the output file.')
    i = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            area = np.array(site['waiting_area'],dtype=np.int32)
            overlay = frame.copy()
            cv2.fillPoly(overlay,[area],(20,170,230))
            frame = cv2.addWeighted(overlay,0.18,frame,0.82,0)
            cv2.polylines(frame,[area],True,(20,170,230),2)
            a,b = [tuple(map(int,p)) for p in site['crossing_line']]
            cv2.line(frame,a,b,(255,150,30),3)
            for tid,obs,state in frames.get(i,[]):
                x1,y1,x2,y2 = map(int,obs['box'])
                colour = (30,210,30) if state['state']=='waiting' else (30,170,255)
                if tid not in eligible:
                    colour = (150,150,150)
                cv2.rectangle(frame,(x1,y1),(x2,y2),colour,2)
                now = i / info['fps']
                observed_total = 0.0
                incomplete = False
                for episode in result['wait_episodes']:
                    if episode['track_id'] != tid or not episode['qualified']:
                        continue
                    if now + 1e-9 < episode['start_seconds'] + rules.min_wait_seconds:
                        continue
                    observed_total += max(0.0, min(now, episode['end_seconds']) - episode['start_seconds'])
                    incomplete |= episode['left_censored']
                    if now >= episode['end_seconds']:
                        incomplete |= episode['right_censored']
                readable = {'outside_waiting_area':'Outside zone', 'moving_in_waiting_area':'Moving',
                            'possible_wait':'Short pause', 'observing':'Observing', 'waiting':'Waiting'}
                label_state = readable.get(state['state'],state['state'])
                if state['state'] != 'waiting' and any(e['track_id']==tid and e['confirmed_seconds']<=now
                                                      for e in result['crossings']):
                    label_state = 'Crossed'
                labels = [f"ID {tid} | {obs['class_name']}",
                          f"{label_state} | Wait {'>=' if incomplete else ''}{observed_total:.2f}s"]
                if state['state']=='possible_wait':
                    labels[1] = f"Short pause {state['observed_wait_seconds']:.2f}s"
                font_scale = max(0.35,min(0.55,info['width']/1700))
                sizes = [cv2.getTextSize(text,cv2.FONT_HERSHEY_SIMPLEX,font_scale,1)[0] for text in labels]
                label_width = max(w for w,h in sizes)+8
                line_height = max(h for w,h in sizes)+8
                lx = min(max(0,x1),max(0,info['width']-label_width))
                ly = max(36,min(y1-2*line_height-4,info['height']-2*line_height-4))
                cv2.rectangle(frame,(lx,ly),(lx+label_width,ly+2*line_height+4),(25,25,25),-1)
                for j,text in enumerate(labels):
                    cv2.putText(frame,text,(lx+4,ly+(j+1)*line_height-3),
                                cv2.FONT_HERSHEY_SIMPLEX,font_scale,(255,255,255),1,cv2.LINE_AA)
                cv2.circle(frame,tuple(map(int,state['smooth_foot'])),4,colour,-1)
            now = i / info['fps']
            waited = {w['track_id'] for w in result['wait_episodes'] if w['qualified'] and
                      w['track_id'] in eligible and w['start_seconds'] + rules.min_wait_seconds <= now}
            crossed = {e['track_id'] for e in result['crossings'] if e['track_id'] in eligible and e['confirmed_seconds'] <= now}
            cv2.rectangle(frame,(0,0),(info['width'],34),(20,20,20),-1)
            cv2.putText(frame,f'{now:.1f}s | waited IDs {len(waited)} | crossed IDs {len(crossed)}',
                        (8,24),cv2.FONT_HERSHEY_SIMPLEX,0.60,(255,255,255),1)
            writer.write(frame)
            i += 1
    finally:
        cap.release()
        writer.release()
    if i != tracked['decoded_frames']:
        raise RuntimeError('Annotated replay decoded a different number of frames.')
    return str(output_path)


## 9. Model location and shared settings
This section uses the included checkpoint when run from a local checkout or a Colab clone.
When the checkout is unavailable, Colab mounts Drive and looks for the existing project weights.
You can set MODEL_PATH to an absolute path to a compatible checkpoint. Upload your video in
Gradio after launching the app.

The one-second waiting rule and ByteTrack settings from the Drive notebook are used here.
Video preparation converts uploads to a constant 30 FPS before you mark the zones.


In [ ]:
import os
import sys
from pathlib import Path

RUNNING_ON_COLAB = "google.colab" in sys.modules
from pathlib import Path

repo_candidates = [Path.cwd(), *Path.cwd().parents]
if RUNNING_ON_COLAB:
    repo_candidates.append(Path("/content/c389-ca1-people-counting"))
repo_root = next(
    (root for root in repo_candidates if (root / "models/improved/best.pt").is_file()),
    None,
)

if repo_root is not None:
    PROJECT_ROOT = repo_root
    MODEL_PATH = repo_root / "models" / "improved" / "best.pt"
    OUTPUT_ROOT = repo_root / "outputs" / "T4_deployment"
else:
    if RUNNING_ON_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = Path("/content/drive/MyDrive/c389 ca1 final (improved model)")
    else:
        PROJECT_ROOT = Path.cwd()
    MODEL_PATH = PROJECT_ROOT / "artifacts" / "L3" / "best.pt"
    OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "T4_deployment"
SHARE_LINK = RUNNING_ON_COLAB

RULES = Rules(
    min_wait_seconds=1.0,
    motion_window_seconds=0.75,
    smoothing_seconds=0.20,
    still_body_heights_per_second=0.06,
    max_observation_gap_seconds=0.25,
    min_track_seconds=0.5,
    min_track_observations=5,
    line_margin_body_heights=0.03,
    line_confirm_seconds=0.15,
)
RULES.validate()
TRACKER_SETTINGS = dict(tracker_type="bytetrack", track_high_thresh=0.25,
                        track_low_thresh=0.10, new_track_thresh=0.25,
                        track_buffer=30, match_thresh=0.80, fuse_score=True)
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Set MODEL_PATH to your trained best.pt. Current path: {MODEL_PATH}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Model:", MODEL_PATH)
print("Results:", OUTPUT_ROOT)
print("Default minimum wait:", RULES.min_wait_seconds, "seconds")


## 10. Interactive deployment code
Run this definition cell once. The user selects a new area and line for each uploaded video.
The waiting table stays at two columns; detailed tracking data is saved in the evidence ZIP.


In [ ]:
"""Interactive T4 video demo. Coordinates and results are isolated per visitor/run."""
from pathlib import Path
from dataclasses import asdict, replace
from datetime import datetime, timezone
from uuid import uuid4
from copy import deepcopy
import json
import math
import shutil
import subprocess
import zipfile
import importlib.metadata

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import gradio as gr
import yaml


AREA_TOOL = "Waiting area — 4 corners"
LINE_TOOL = "Crossing line — 2 endpoints"
WAITING_COLUMNS = ["Track ID", "Waiting time (s)"]
DEFAULT_TRACKER = dict(tracker_type="bytetrack", track_high_thresh=0.25,
                       track_low_thresh=0.10, new_track_thresh=0.25,
                       track_buffer=30, match_thresh=0.80, fuse_score=True)


def waiting_table(result, show_zero=False):
    """Two display columns; incomplete observed waits remain explicit lower bounds."""
    rows = []
    for person in result["people"]:
        if not person["eligible"]:
            continue
        if not person["waited"] and not show_zero:
            continue
        value = f"{person['observed_wait_seconds']:.2f}"
        if person["waited"] and person["duration_incomplete"]:
            value = ">= " + value
        rows.append({"Track ID": int(person["track_id"]), "Waiting time (s)": value})
    return pd.DataFrame(rows, columns=WAITING_COLUMNS)


def bundle_evidence(run_dir):
    """Refresh the download after evaluation or failure examples are added."""
    run_dir = Path(run_dir)
    archive = run_dir / "results_and_evidence.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
        for file in sorted(run_dir.rglob("*")):
            if file.is_file() and file.name not in (archive.name, "annotated_raw.mp4"):
                zipped.write(file, file.relative_to(run_dir))
    return archive


def ffmpeg_executable():
    executable = shutil.which("ffmpeg")
    if executable:
        return executable
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except (ImportError, RuntimeError) as exc:
        raise ValueError("Install imageio-ffmpeg by rerunning the notebook setup cell.") from exc


def transcode_video(source, destination, normalize=False):
    """Always return browser-playable H.264; normalisation precedes coordinate selection."""
    command = [ffmpeg_executable(), "-y", "-loglevel", "error", "-i", str(source),
               "-map", "0:v:0", "-an"]
    if normalize:
        command += ["-vf", "scale='trunc(min(1280,iw)/2)*2':-2,fps=30", "-fps_mode", "cfr"]
    command += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "22",
                "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(destination)]
    try:
        completed = subprocess.run(command, capture_output=True, text=True, timeout=180)
    except subprocess.TimeoutExpired as exc:
        raise ValueError("Video conversion took too long. Try a shorter clip exported as MP4.") from exc
    if completed.returncode or not Path(destination).is_file():
        raise ValueError("This video could not be converted. Try exporting it as MP4 from your phone.")
    return Path(destination)


def _segments_cross(a, b, c, d):
    def orient(p, q, r):
        return (q[0]-p[0])*(r[1]-p[1]) - (q[1]-p[1])*(r[0]-p[0])
    def on(p, q, r):
        return (abs(orient(p,q,r)) < 1e-8 and min(p[0],q[0]) <= r[0] <= max(p[0],q[0])
                and min(p[1],q[1]) <= r[1] <= max(p[1],q[1]))
    x,y,z,w = orient(a,b,c),orient(a,b,d),orient(c,d,a),orient(c,d,b)
    return (x*y < 0 and z*w < 0) or on(a,b,c) or on(a,b,d) or on(c,d,a) or on(c,d,b)


def validate_selection(state):
    if not state or not Path(state.get("prepared_path", "")).is_file():
        raise ValueError("Upload a video and click Load video first.")
    area, line = state["area"], state["line"]
    if len(area) != 4 or len(line) != 2:
        raise ValueError("Mark four waiting-area corners and two crossing-line endpoints.")
    if len({tuple(p) for p in area}) != 4:
        raise ValueError("The waiting area's four corners must be different.")
    if _segments_cross(area[0],area[1],area[2],area[3]) or _segments_cross(area[1],area[2],area[3],area[0]):
        raise ValueError("The waiting-area edges cross. Clear the area and click corners around its edge.")
    site = {"site_id":state["token"], "frame_width":state["info"]["width"],
            "frame_height":state["info"]["height"], "waiting_area":area,
            "crossing_line":line, "crossing_direction":"both",
            "recorded_on":None, "camera_position":"Selected by the user on the uploaded video"}
    validate_site(site,state["info"]["width"],state["info"]["height"])
    return site


def selection_status(state):
    if not state:
        return "Upload a video, then click **Load video**."
    a,b = len(state["area"]),len(state["line"])
    if a == 4 and b == 2:
        try:
            validate_selection(state)
        except ValueError as exc:
            return str(exc)
        return "**Ready.** Check the yellow waiting area and blue crossing line, then process the video."
    return f"Waiting area: **{a}/4** corners. Crossing line: **{b}/2** endpoints. Click points around the ground area."


def draw_selection(state):
    if not state:
        return None
    picture = Image.fromarray(state["frame_rgb"]).convert("RGBA")
    layer = Image.new("RGBA",picture.size,(0,0,0,0))
    draw = ImageDraw.Draw(layer)
    area,line = [tuple(p) for p in state["area"]],[tuple(p) for p in state["line"]]
    width = max(2,picture.width//350)
    if len(area)==4:
        draw.polygon(area,fill=(255,190,0,55),outline=(255,190,0,255),width=width)
    elif len(area)>1:
        draw.line(area,fill=(255,190,0,255),width=width)
    if len(line)==2:
        draw.line(line,fill=(0,165,255,255),width=width+1)
    for label,points,colour in [("W",area,(255,190,0,255)),("C",line,(0,165,255,255))]:
        for i,(x,y) in enumerate(points,1):
            draw.ellipse((x-5,y-5,x+5,y+5),fill=colour)
            draw.text((x+7,y-12),f"{label}{i}",fill=colour,stroke_width=1,stroke_fill=(0,0,0,255))
    return np.array(Image.alpha_composite(picture,layer).convert("RGB"))


class T4Service:
    def __init__(self, model_path, output_root, rules=None, tracker_settings=None, track_fn=None):
        self.model_path = Path(model_path)
        self.output_root = Path(output_root)
        self.output_root.mkdir(parents=True,exist_ok=True)
        self.rules = rules or Rules(min_wait_seconds=1.0)
        self.rules.validate()
        self.tracker_settings = dict(tracker_settings or DEFAULT_TRACKER)
        self.track_fn = track_fn or track_video

    def prepare(self, path):
        if not path:
            raise ValueError("Choose a video file first.")
        path = Path(path)
        if not path.is_file() or path.stat().st_size == 0:
            raise ValueError("The uploaded video is empty or unavailable. Upload it again.")
        if path.suffix.lower() not in (".mp4",".mov",".m4v",".avi",".mkv"):
            raise ValueError("Upload an MP4, MOV, M4V, AVI or MKV video.")
        if path.stat().st_size > 350*1024*1024:
            raise ValueError("Please use a clip smaller than 350 MB.")
        original_info = video_info(path)
        if original_info["duration_seconds"] > 120:
            raise ValueError("Please use a clip no longer than two minutes.")
        token = uuid4().hex
        cache = self.output_root/"_uploads"/token
        cache.mkdir(parents=True,exist_ok=False)
        prepared = transcode_video(path,cache/"prepared.mp4",normalize=True)
        info = video_info(prepared)
        if 1/info["fps"] > self.rules.max_observation_gap_seconds:
            raise ValueError("Video frame rate is incompatible with the selected waiting rules.")
        return {"token":token,"prepared_path":str(prepared),"original_name":path.name,
                "original_sha256":sha256_file(path),"original_info":original_info,
                "info":info,"frame_rgb":cv2.cvtColor(read_video_frame(prepared),cv2.COLOR_BGR2RGB),
                "area":[],"line":[],"history":[],"frame_seconds":0.0}

    def mark(self, state, point, tool):
        if not state:
            raise ValueError("Load a video before selecting points.")
        state = deepcopy(state)
        if len(point)!=2 or not all(math.isfinite(float(v)) for v in point):
            raise ValueError("Choose a valid point on the image.")
        x,y = [int(round(float(v))) for v in point]
        if not (0<=x<state["info"]["width"] and 0<=y<state["info"]["height"]):
            raise ValueError("Choose a point inside the image.")
        if tool not in (AREA_TOOL,LINE_TOOL):
            raise ValueError("Choose which region to mark.")
        key,limit = ("area",4) if tool==AREA_TOOL else ("line",2)
        if len(state[key])>=limit:
            raise ValueError("Those points are complete. Use Undo or clear that region to change them.")
        if [x,y] in state[key]:
            raise ValueError("Choose a different point for each corner or endpoint.")
        state[key].append([x,y])
        state["history"].append(key)
        next_tool = LINE_TOOL if len(state["area"])==4 else AREA_TOOL
        return state,next_tool

    def undo(self,state):
        if not state:
            return None,AREA_TOOL
        state=deepcopy(state)
        if state["history"]:
            key=state["history"].pop()
            state[key].pop()
        return state,AREA_TOOL if len(state["area"])<4 else LINE_TOOL

    def clear(self,state,key):
        if not state:
            return None,AREA_TOOL
        state=deepcopy(state)
        state[key]=[]
        state["history"]=[entry for entry in state["history"] if entry!=key]
        return state,AREA_TOOL if len(state["area"])<4 else LINE_TOOL

    def change_frame(self,state,seconds):
        if not state:
            raise ValueError("Load a video first.")
        state=deepcopy(state)
        frame=min(state["info"]["frames"]-1,max(0,round(float(seconds)*state["info"]["fps"])))
        state["frame_rgb"]=cv2.cvtColor(read_video_frame(state["prepared_path"],frame),cv2.COLOR_BGR2RGB)
        state["frame_seconds"]=frame/state["info"]["fps"]
        return state

    def process(self,state,min_wait,show_zero=False,progress=None):
        site=validate_selection(state)
        rules=replace(self.rules,min_wait_seconds=float(min_wait))
        rules.validate()
        if not self.model_path.is_file():
            raise ValueError("The trained model is missing. Check MODEL_PATH in notebook section 9.")
        if not 0<float(self.tracker_settings.get("track_low_thresh",0))<=1:
            raise ValueError("Tracker settings are invalid. Check the notebook configuration.")
        run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid4().hex[:8]
        run_dir=self.output_root/"runs"/run_id
        run_dir.mkdir(parents=True,exist_ok=False)
        tracker_path=run_dir/"bytetrack.yaml"
        tracker_path.write_text(yaml.safe_dump(self.tracker_settings),encoding="utf-8")
        manifest={"run_id":run_id,"created_utc":datetime.now(timezone.utc).isoformat(),
                  "original_name":state["original_name"],"original_sha256":state["original_sha256"],
                  "original_info":state["original_info"],"prepared_path":state["prepared_path"],
                  "prepared_sha256":sha256_file(state["prepared_path"]),"clip_info":state["info"],
                  "normalization":"H.264, constant 30 FPS; selections and timestamps use this prepared video",
                  "model_path":str(self.model_path),"model_sha256":sha256_file(self.model_path),
                  "rules":asdict(rules),"site":site,"tracker_settings":self.tracker_settings,
                  "detection_confidence":0.10,"image_size":640,
                  "camera_stability":"Not automatically verified; geometry remains fixed in image coordinates",
                  "minimum_wait_is_user_selected":True}
        manifest["package_versions"] = {}
        for package in ("ultralytics", "gradio", "imageio-ffmpeg", "numpy", "pandas"):
            try:
                manifest["package_versions"][package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                manifest["package_versions"][package] = "not installed"
        manifest["opencv_version"] = cv2.__version__
        save_json(run_dir/"run_manifest.json",manifest)
        if progress:progress(0.05,desc="Tracking people")
        tracked=self.track_fn(state["prepared_path"],self.model_path,tracker_path,
                              confidence=0.10,image_size=640,
                              progress=(lambda done,total:progress(0.05+0.60*done/max(total,1),desc="Tracking people"))
                              if progress else None)
        save_json(run_dir/"tracks.json",tracked)
        result=analyze_tracks(tracked["tracks"],site,rules)
        save_json(run_dir/"analysis.json",result)
        save_json(run_dir/"summary.json",result["summary"])
        save_json(run_dir/"site.json",site)
        save_json(run_dir/"rules.json",asdict(rules))
        table=waiting_table(result,show_zero)
        table.to_csv(run_dir/"waiting_times.csv",index=False)
        save_csv(run_dir/"people_details.csv",result["people"],fields=[
            "track_id","eligible","outcome","waited","crossed","observed_wait_seconds",
            "wait_episode_count","duration_incomplete","first_crossing_seconds","first_seen_seconds",
            "last_seen_seconds","observations","majority_detector_class","flags"])
        save_csv(run_dir/"wait_episodes.csv",result["wait_episodes"],fields=[
            "track_id","start_seconds","end_seconds","seconds","qualified","incomplete",
            "left_censored","right_censored","end_reason"])
        save_csv(run_dir/"crossings.csv",result["crossings"],fields=[
            "track_id","seconds","confirmed_seconds","direction","screen_direction"])
        save_csv(run_dir/"ground_truth.csv",[],fields=[
            "human_id","waited","crossed","wait_seconds","duration_complete","notes"])
        save_csv(run_dir/"id_mapping.csv",[],fields=["human_id","track_id","notes"])
        save_csv(run_dir/"failure_review.csv",[],fields=[
            "human_id","track_id","start_seconds","end_seconds","stage","description","evidence_file"])
        Image.fromarray(draw_selection(state)).save(run_dir/"marked_frame.png")
        if progress:progress(0.70,desc="Drawing results")
        raw=run_dir/"annotated_raw.mp4"
        render_video(state["prepared_path"],tracked,result,site,rules,raw)
        playable=transcode_video(raw,run_dir/"annotated.mp4")
        if progress:progress(0.95,desc="Saving evidence")
        archive=bundle_evidence(run_dir)
        if progress:progress(1.0,desc="Complete")
        summary=result["summary"]
        counters=(f"**Waited:** {summary['waited_track_ids']}  ·  "
                  f"**Crossed:** {summary['crossed_track_ids']}  ·  "
                  f"**Waited then crossed:** {summary['exclusive_outcomes']['waited_then_crossed']}")
        status="Complete. Waiting time is observed stationary time inside the selected area."
        if not summary["waited_track_ids"]:
            status+=f" No ID reached the {rules.min_wait_seconds:g}-second waiting rule."
        return {"video":str(playable),"table":table,"counters":counters,"status":status,
                "csv":str(run_dir/"waiting_times.csv"),"archive":str(archive),"run_dir":str(run_dir)}


def build_demo(service):
    """One app for the team; every browser session has its own video and selections."""
    def drawing_outputs(state,tool):
        ready=False
        if state and len(state["area"])==4 and len(state["line"])==2:
            try:
                validate_selection(state)
                ready=True
            except ValueError:
                pass
        return (state,draw_selection(state),selection_status(state),tool,gr.update(interactive=ready),
                None,pd.DataFrame(columns=WAITING_COLUMNS),"",None,None)

    def load_video(path,progress=gr.Progress()):
        try:
            progress(0,desc="Preparing video")
            state=service.prepare(path)
            progress(1,desc="Ready to mark")
            return (*drawing_outputs(state,AREA_TOOL),state["prepared_path"],
                    gr.update(maximum=max(0.01,(state["info"]["frames"]-1)/state["info"]["fps"]),value=0))
        except (ValueError,OSError) as exc:
            raise gr.Error(str(exc)) from exc

    def mark_click(state,tool,evt:gr.SelectData):
        try:
            return drawing_outputs(*service.mark(state,evt.index,tool))
        except ValueError as exc:
            raise gr.Error(str(exc)) from exc

    def mark_coordinates(state,tool,x,y):
        try:
            return drawing_outputs(*service.mark(state,(x,y),tool))
        except (ValueError,TypeError) as exc:
            raise gr.Error(str(exc)) from exc

    def undo(state):
        return drawing_outputs(*service.undo(state))

    def clear_area(state):
        return drawing_outputs(*service.clear(state,"area"))

    def clear_line(state):
        return drawing_outputs(*service.clear(state,"line"))

    def change_frame(state,second):
        try:
            state=service.change_frame(state,second)
            return state,draw_selection(state)
        except ValueError as exc:
            raise gr.Error(str(exc)) from exc

    def process(state,min_wait,show_zero,progress=gr.Progress()):
        try:
            case=service.process(state,min_wait,show_zero,progress)
            return case["video"],case["table"],case["counters"],case["status"],case["csv"],case["archive"]
        except (ValueError,OSError,RuntimeError) as exc:
            raise gr.Error(str(exc)) from exc

    with gr.Blocks(title="People Counting Assistant — T4",analytics_enabled=False) as demo:
        gr.Markdown("# People Counting Assistant\n### Wait or cross\n"
                    "Upload a video, mark where people wait, then view the waiting time for each tracked ID.")
        session=gr.State(None)
        with gr.Row():
            with gr.Column(scale=1,min_width=280):
                uploaded=gr.File(label="1. Upload your video",type="filepath",
                                 file_types=[".mp4",".mov",".MOV",".m4v",".avi",".mkv"])
                load=gr.Button("Load video",variant="primary")
                preview=gr.Video(label="Video preview",interactive=False)
                min_wait=gr.Number(label="Minimum wait (seconds)",value=service.rules.min_wait_seconds,
                                   minimum=service.rules.motion_window_seconds,maximum=30,step=0.25,precision=2)
                show_zero=gr.Checkbox(label="Also show IDs with no qualifying wait",value=False)
            with gr.Column(scale=2,min_width=420):
                tool=gr.Radio([AREA_TOOL,LINE_TOOL],value=AREA_TOOL,label="2. Choose what to mark")
                scene=gr.Image(label="Click corners around the waiting area, then the two line endpoints",
                               type="numpy",interactive=False,height=440)
                frame_time=gr.Slider(0,0.01,value=0,step=0.1,label="Frame used for marking (seconds)")
                with gr.Row():
                    undo_button=gr.Button("Undo last point")
                    clear_area_button=gr.Button("Clear waiting area")
                    clear_line_button=gr.Button("Clear crossing line")
                with gr.Accordion("Enter a point by coordinates instead",open=False):
                    with gr.Row():
                        x=gr.Number(label="X",value=0,precision=0)
                        y=gr.Number(label="Y",value=0,precision=0)
                        add_point=gr.Button("Add point")
                status=gr.Markdown(selection_status(None))
                process_button=gr.Button("3. Process video",variant="primary",interactive=False)
        counters=gr.Markdown()
        with gr.Row():
            result_video=gr.Video(label="Annotated result",interactive=False)
            result_table=gr.Dataframe(headers=WAITING_COLUMNS,datatype=["number","str"],
                                      interactive=False,label="Observed waiting time by ID")
        gr.Markdown("`>=` means the full wait was not observed. Only qualifying waits are shown by default. "
                    "IDs are tracking labels and may have gaps; ID changes can split one person's result. "
                    "Keep the camera steady so the marked area stays on the same ground.")
        with gr.Row():
            csv_download=gr.File(label="Download waiting-time table",interactive=False)
            evidence_download=gr.File(label="Download results and evidence",interactive=False)
        draw_out=[session,scene,status,tool,process_button,result_video,result_table,counters,
                  csv_download,evidence_download]
        load.click(load_video,inputs=[uploaded],outputs=draw_out+[preview,frame_time],api_name="load_video")
        scene.select(mark_click,inputs=[session,tool],outputs=draw_out,queue=False,api_name=False)
        add_point.click(mark_coordinates,inputs=[session,tool,x,y],outputs=draw_out,
                        queue=False,api_name="add_point")
        undo_button.click(undo,inputs=[session],outputs=draw_out,queue=False,api_name="undo_point")
        clear_area_button.click(clear_area,inputs=[session],outputs=draw_out,queue=False,api_name="clear_area")
        clear_line_button.click(clear_line,inputs=[session],outputs=draw_out,queue=False,api_name="clear_line")
        frame_time.release(change_frame,inputs=[session,frame_time],outputs=[session,scene],
                           queue=False,api_name="marking_frame")
        process_button.click(process,inputs=[session,min_wait,show_zero],outputs=[result_video,result_table,
                             counters,status,csv_download,evidence_download],
                             concurrency_limit=1,concurrency_id="t4_processing",api_name="process_video")
    demo.queue(default_concurrency_limit=1)
    return demo


## 11. Launch Gradio
Open the link shown below. **Load video → mark four waiting-area corners → mark two line
endpoints → Process video.** Use Undo/Clear to correct points. You can move the marking-frame
slider to see a person waiting before selecting the ground area.

The camera should stay steady: a moving background makes a fixed pixel area drift.
Use short clips (up to two minutes / 350 MB). Keep Colab running while demonstrating.


In [ ]:
if "demo" in globals():
    demo.close()
service = T4Service(MODEL_PATH, OUTPUT_ROOT, rules=RULES, tracker_settings=TRACKER_SETTINGS)
demo = build_demo(service)
demo.launch(share=SHARE_LINK, show_error=True, max_file_size="350mb",
            allowed_paths=[str(OUTPUT_ROOT.resolve())])


## 12. Select a completed run for notebook review (optional)
Run this after processing a video in Gradio. Leave `RUN_FOLDER` empty for the latest completed
run, or paste a particular run folder. This loads saved results and does not run the model again.


In [ ]:
RUN_FOLDER = ""
candidates = sorted((OUTPUT_ROOT / "runs").glob("*/analysis.json"))
if RUN_FOLDER:
    OUT_DIR = Path(RUN_FOLDER)
elif candidates:
    OUT_DIR = candidates[-1].parent
else:
    OUT_DIR = None

RESULT = None
if OUT_DIR is None:
    print("Process a video in Gradio first, then rerun this cell.")
else:
    RESULT = json.loads((OUT_DIR / "analysis.json").read_text(encoding="utf-8"))
    MANIFEST = json.loads((OUT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
    print("Selected run:", OUT_DIR)
    print("Video:", MANIFEST["original_name"])
    print("Minimum wait:", MANIFEST["rules"]["min_wait_seconds"], "seconds")


## 13. Simple waiting-time table (optional)
The same two-column view appears in Gradio. Turn on `SHOW_ZERO_WAITS` to include eligible IDs
with no qualifying wait. These values sum qualifying stationary episodes, not all time in the zone.


In [ ]:
SHOW_ZERO_WAITS = False
if RESULT is not None:
    display(waiting_table(RESULT, show_zero=SHOW_ZERO_WAITS))
else:
    print("Select a completed run in section 12 first.")


## 14. Replay the saved result (optional)
This is the same annotated result shown in Gradio.


In [ ]:
from IPython.display import Video, display
if OUT_DIR is not None:
    display(Video(str(OUT_DIR / "annotated.mp4"), embed=True, width=900))
else:
    print("Select a completed run in section 12 first.")


## 15. Manual test labels (optional, useful for your evaluation evidence)
Inspect the video and fill the saved CSVs using a spreadsheet editor. Label **every relevant
person**, including people missed by the model. A human ID is your manual label; a track ID
is the number drawn by the program. One human may map to several fragmented track IDs.
Leave their mapping blank if missed. Use the same minimum-wait rule as the saved run.

`ground_truth.csv`: human_id, waited (true/false), crossed (true/false), wait_seconds,
duration_complete (true/false), notes. Measure wait_seconds by video timestamps; use false
for duration_complete when the start/end is outside the clip or obscured.

`id_mapping.csv`: human_id, track_id, notes. Add one row per matching track ID.
`failure_review.csv`: record missed detections, ID switches, wrong area/movement decisions,
or crossing errors. These templates are created once per run and are not overwritten here.


In [ ]:
if OUT_DIR is not None:
    for name in ["ground_truth.csv", "id_mapping.csv", "failure_review.csv"]:
        print(name, "→", OUT_DIR / name)
else:
    print("Select a completed run in section 12 first.")


## 16. Evaluate against your manual labels (optional)
This checks waiting/crossing count errors and duration error. A duration MAE of None means
there were no complete matched waits to evaluate; it does not mean zero error. The saved
JSON includes misses, extra IDs, fragmentation and duration coverage. Choose settings using
tuning clips, then keep them fixed on separate held-out clips used for your final evidence.


In [ ]:
RUN_MANUAL_EVALUATION = False  # Set True after filling the two CSVs in section 15.
if RUN_MANUAL_EVALUATION and RESULT is not None:
    def read_rows(path):
        with open(path, newline="", encoding="utf-8-sig") as handle:
            return list(csv.DictReader(handle))
    evaluation = evaluate_people(RESULT, read_rows(OUT_DIR / "ground_truth.csv"),
                                  read_rows(OUT_DIR / "id_mapping.csv"))
    save_json(OUT_DIR / "evaluation.json", evaluation)
    save_csv(OUT_DIR / "evaluation_people.csv", evaluation["people"])
    metrics = evaluation["metrics"]
    display(pd.DataFrame([{
        "Wait count error": metrics["waited_raw_count_absolute_error"],
        "Cross count error": metrics["crossed_raw_count_absolute_error"],
        "Wait duration MAE (s)": metrics["duration_mae_seconds"],
        "Complete waits evaluated": metrics["duration_evaluated_people"],
    }]))
    print("Updated download:", bundle_evidence(OUT_DIR))
else:
    print("Optional: complete section 15, select the run, then set RUN_MANUAL_EVALUATION=True.")


## 19. Save failure examples and refresh the evidence ZIP (optional)
The old sections 17–18 have been removed. Keep screenshots of useful successes and failures
for your report. Enter video timestamps and short filenames below; the prepared-video timestamps
match the Gradio output. Add explanations to failure_review.csv. This cell refreshes the ZIP
after labels/evaluation/screenshots change. Rerun Process in Gradio only for a new experiment.


In [ ]:
EXAMPLES = []  # Example: [(12.5, "id_switch"), (18.0, "wait_then_cross")]
if OUT_DIR is not None:
    prepared_path = Path(MANIFEST["prepared_path"])
    for seconds, label in EXAMPLES:
        if not math.isfinite(float(seconds)) or float(seconds) < 0:
            raise ValueError("Use a nonnegative video timestamp.")
        safe_label = "".join(c if c.isalnum() or c in "_-" else "_" for c in str(label))[:60]
        image = read_video_frame(OUT_DIR / "annotated.mp4",
                                 round(float(seconds) * MANIFEST["clip_info"]["fps"]))
        evidence_dir = OUT_DIR / "examples"
        evidence_dir.mkdir(exist_ok=True)
        cv2.imwrite(str(evidence_dir / f"{float(seconds):.2f}s_{safe_label}.jpg"), image)
    print("Evidence ZIP:", bundle_evidence(OUT_DIR))
else:
    print("Select a completed run in section 12 first.")
